# 01 - Problem Setup and EDA

In this notebook we:
-  Load a small synthetic water demand dataset with multiple stations, each having instantaneous flow rates measured at 15 minute intervals
- Explore station level behavior
- Apply the cleaning + feature engineering pipeline:
   - Wide → long format (timestamp, station, consumption).
   - Quality flags (long gaps, structural zeros, small-gap interpolation).
   - Leak / extreme-spike detection and clamping.  
- Save a processed dataframe ready for model training and artifact replay


In [ ]:
from pathlib import Path

DATASET_CANDIDATES = ("df_stationsv3.csv", "df_stations_v3.csv")


def resolve_project2_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    for base in candidates:
        if base.name == "project2" and any((base / name).exists() for name in DATASET_CANDIDATES):
            return base
        nested = base / "project2"
        if any((nested / name).exists() for name in DATASET_CANDIDATES):
            return nested
    raise FileNotFoundError(
        "Could not locate project2 root. Run this notebook from GestaguaOG or project2."
    )


def resolve_raw_csv_path(project_root: Path) -> Path:
    for name in DATASET_CANDIDATES:
        candidate = project_root / name
        if candidate.exists():
            return candidate
    raise FileNotFoundError("Could not find a public demo dataset CSV in the project root.")


PROJECT2_ROOT = resolve_project2_root()
RAW_CSV_PATH = resolve_raw_csv_path(PROJECT2_ROOT)
PROCESSED_DIR = PROJECT2_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT2_ROOT}")
print(f"Raw CSV path: {RAW_CSV_PATH}")
print(f"Processed dir: {PROCESSED_DIR}")


In [ ]:
import pandas as pd

csv_path = RAW_CSV_PATH
df_raw = pd.read_csv(csv_path)
df_raw = df_raw.loc[:, ~df_raw.columns.str.startswith("Unnamed")]
df_raw.head()

## Data Visualizations

- One row per (timestamp, station)
- A single 'consumption' column indicating the instantaneous flow rate

In [ ]:
df_long = df_raw.melt(
    id_vars="timestamp",
    var_name="station",
    value_name="consumption"
)
df_long["timestamp"] = pd.to_datetime(df_long["timestamp"], utc=True, errors="coerce")
print(df_long["station"].value_counts())
df_long.head()


In [ ]:
from io import BytesIO

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from IPython.display import Image, display


def render_figure(fig):
    buf = BytesIO()
    fig.savefig(buf, format="png", bbox_inches="tight", dpi=150)
    buf.seek(0)
    display(Image(data=buf.getvalue()))
    plt.close(fig)


def plot_window(df_long, stations, start, end, title=None):
    start_ts = pd.to_datetime(start, utc=True)
    end_ts   = pd.to_datetime(end, utc=True)

    mask = (
        (df_long["timestamp"] >= start_ts) &
        (df_long["timestamp"] <= end_ts) &
        (df_long["station"].isin(stations if isinstance(stations, list) else [stations]))
    )

    tmp = df_long.loc[mask].copy()

    fig, ax = plt.subplots(figsize=(12, 5))
    for st, g in tmp.groupby("station"):
        ax.plot(g["timestamp"], g["consumption"], label=st)
    ax.set_xlabel("Timestamp (UTC)")
    ax.set_ylabel("Instantaneous flow rate")
    if title:
        ax.set_title(title)
    handles, labels = ax.get_legend_handles_labels()
    if labels:
        ax.legend()
    else:
        ax.text(0.5, 0.5, "No data in selected window", ha="center", va="center", transform=ax.transAxes)
    ax.grid(True)
    plt.tight_layout()
    render_figure(fig)


example_stations = [
    "Station_9",
    "Station_7",
]

for station in example_stations:
    plot_window(
        df_long,
        stations=[station],
        start="2024-01-01",
        end="2024-01-08",
        title="7-day window: different regimes and variability",
    )


## Heteroscedasticity Analysis: variance vs level

Water demand is typically **heteroscedastic**:

- Periods or stations with higher average flow also tend to show higher variability.
- This matters for forecasting because prediction intervals must widen as the expected level increases.


To quantify this, we:
1. Aggregate to daily statistics per station:
   - Daily mean flow μ (level).
   - Daily standard deviation σ (volatility).
   - Coefficient of variation `CV = σ / μ` (for μ > 0).
2. Visualize `σ` vs `μ` across all stations and days.
3. Summarize `CV` per station to compare how “noisy” each station is relative to its own level.

In [ ]:
df_long["timestamp"] = pd.to_datetime(df_long["timestamp"], utc=True, errors="coerce")

df_long["date"] = df_long["timestamp"].dt.date

daily_stats = (
    df_long
    .groupby(["station", "date"])["consumption"]
    .agg(mean="mean", std="std")
    .reset_index()
)

daily_stats = daily_stats[daily_stats["mean"] > 0].copy()
eps = 1e-6
daily_stats["cv"] = daily_stats["std"] / (daily_stats["mean"] + eps)

cv_by_station = (
    daily_stats
    .groupby("station")
    .agg(
        mean_daily_flow=("mean", "mean"),
        mean_daily_std=("std", "mean"),
        median_cv=("cv", "median"),
        n_days=("cv", "size"),
    )
    .reset_index()
)

display(daily_stats.head())
display(cv_by_station.head())

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
sc = ax.scatter(
    daily_stats["mean"],
    daily_stats["std"],
    c=daily_stats["cv"],
    s=12,
    alpha=0.6,
    norm=mcolors.LogNorm(),   # CV often spans orders of magnitude
)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Daily mean flow")
ax.set_ylabel("Daily standard deviation of flow")
ax.set_title("Heteroscedasticity: absolute vs relative volatility")
cbar = fig.colorbar(sc, ax=ax)
cbar.set_label("Coefficient of variation (std / mean)")
ax.grid(True, which="both", ls="--", alpha=0.3)
fig.tight_layout()
render_figure(fig)


Each dot represents one day for one station
- X-axis (log): mean daily consumption/flow
- Y-axis (log): daily standard deviation of consumption/flow
- Color: Coefficient of variation (CV = std/mean) relative volatility

Most points lie on an increasing band, days with higher average flow tend to have higher absolute variability (std). The figure showcases this closely as a straight line on a log-log scale, an indication of heteroscedastic behavior in demand time series.

Small/medium stations (around left-middle of x-axis) are often colored green/yellow, signaling a high CV, they fluctuate relative to their mean

Large stations (right side) are more blue/purple, signaling a lower CV. Absolute std is high (big flows), but relative to the mean they are more stable.

A Single homoscedastic model would underestimate uncertainty for small stations and overestinate for large, stable ones.

This justifies the design choice of Mondrian conformal intervals, that adapt to station and time of day


## Periodicity Analysis: Daily shape stability score

To quantify how periodic or repeatable each station’s pattern is, we use Seasonal Trend Loess Decomposition (STL)

For each station we take the full 15-minute time series and decompose it as:

- **Trend**: slow changes over weeks/months.
- **Seasonal component**: a **repeating daily pattern** with period = 96 steps  
  (96 × 15 minutes = 24 hours).
- **Remainder**: whatever is left (noise, irregular events, shifts, anomalies).

A higher STL strength indicates variance is mostly in the seasonal component
- Cleaner "template" day
- Easier to forecast

Low STL strength indicates variance is mostly in the remainder
- Seasonal curve tends to be less pronounced or unstable
- Remainder catches spikes or abrupts level shifts


In [ ]:
import numpy as np
import pandas as pd

try:
    from statsmodels.tsa.seasonal import STL
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "statsmodels is required for STL analysis. Install it with `pip install statsmodels`."
    ) from exc


In [ ]:
def stl_periodicity_score(
    s: pd.Series,
    freq="15min",
    period=96,
    max_nan_frac=0.3,
) -> float:

    s = s.dropna().sort_index()
    if s.empty:
        return np.nan

    if not isinstance(s.index, pd.DatetimeIndex):
        s.index = pd.to_datetime(s.index, utc=True, errors="coerce")
        s = s[~s.index.isna()]
        if s.empty:
            return np.nan

    s = s.asfreq(freq)

    if s.isna().mean() > max_nan_frac:
        return np.nan
    if s.std() == 0:
        return np.nan

    s = s.interpolate(limit=4)
    stl = STL(s, period=period, robust=True)
    res = stl.fit()

    seasonal = res.seasonal
    remainder = res.resid

    var_seasonal = float(np.var(seasonal))
    var_remainder = float(np.var(remainder))
    denom = var_seasonal + var_remainder

    if denom <= 0:
        return np.nan

    strength = var_seasonal / denom
    return strength


def plot_stl_for_station(
    df_long,
    station,
    freq="15min",
    period=96,          # 96 x 15min = 1 day
    start=None,
    end=None,
    title_prefix=None,
):
    g = df_long[df_long["station"] == station].copy()
    if g.empty:
        print(f"No data for {station}")
        return

    g = g.sort_values("timestamp")

    # Ensure timestamp is tz-aware UTC
    g["timestamp"] = pd.to_datetime(g["timestamp"], utc=True, errors="coerce")
    g = g[~g["timestamp"].isna()]

    s = g.set_index("timestamp")["consumption"]

    # Make start/end also tz-aware UTC so comparisons are valid
    if start is not None:
        start_ts = pd.to_datetime(start, utc=True)
        s = s[s.index >= start_ts]
    if end is not None:
        end_ts = pd.to_datetime(end, utc=True)
        s = s[s.index <= end_ts]

    if s.empty:
        print(f"No data for {station} in selected window")
        return

    # ensure regular grid
    s = s.asfreq(freq)

    # light interpolation for gaps
    s = s.interpolate(limit=4)

    stl = STL(s, period=period, robust=True)
    res = stl.fit()

    fig = res.plot()
    fig.set_size_inches(10, 6)
    if title_prefix is None:
        title_prefix = "STL decomposition"
    fig.suptitle(f"{title_prefix} – {station}", fontsize=14)
    fig.tight_layout()
    render_figure(fig)

df_long["timestamp"] = pd.to_datetime(df_long["timestamp"], utc=True, errors="coerce")

stl_by_station = (
    df_long
      .set_index("timestamp")
      .groupby("station")["consumption"]
      .apply(stl_periodicity_score)
      .rename("stl_daily_strength")
      .reset_index()
)

In [ ]:
station_regimes = cv_by_station.merge(stl_by_station, on="station", how="left")
# Plot regimes: volatility vs periodicity
fig, ax = plt.subplots(figsize=(7, 5))

size_scale = station_regimes["mean_daily_flow"] / station_regimes["mean_daily_flow"].max()
sizes = 30 + 120 * size_scale  # bigger stations → bigger points

sc = ax.scatter(
    station_regimes["median_cv"],
    station_regimes["stl_daily_strength"],
    s=sizes,
    alpha=0.8,
)

ax.set_xscale("linear")
ax.set_xlabel("Median daily coefficient of variation (CV)")
ax.set_ylabel("STL daily seasonal strength (0–1)")
ax.set_title("Station regimes: within-day volatility vs daily periodicity")
ax.grid(True, alpha=0.3)

top_cv = station_regimes.sort_values("median_cv", ascending=False).head(2)
top_stl = station_regimes.sort_values("stl_daily_strength", ascending=False).head(2)

to_annotate = (
    pd.concat([top_cv, top_stl], ignore_index=True)
      .drop_duplicates(subset="station")
)

for _, row in to_annotate.iterrows():
    ax.text(
        row["median_cv"],
        row["stl_daily_strength"],
        row["station"],
        fontsize=8,
        ha="left",
        va="bottom",
        alpha=0.8,
    )

fig.tight_layout()
render_figure(fig)
station_regimes

STL strength

In [ ]:
plot_stl_for_station(df_long, "Station_10", start="2025-02-08", end="2025-02-21")

In [ ]:
plot_stl_for_station(df_long, "Station_1", start="2024-01-01", end="2024-01-15")

Figure compares Station_10 which has a STL strength score of 1, and Station_1 whith a score of 0.357446.

Im going to leave this for validation


## Preprocessing

The goal of this section is to build a **model-agnostic, cleaned time series**
that any forecasting model can consume. We do this with the function
`preprocess`, which applies four main steps:

 **Type normalisation and sorting**

   - Convert `timestamp` to a timezone-aware `datetime` in UTC.
   - Force `station` to string and `consumption` to numeric.
   - Sort by `(station, timestamp)` to get a well-ordered panel.

 **Leak / spike cleaning per station**

   For each station independently we detect extreme positive spikes:

   - Compute a rolling mean and standard deviation over a 96-step window
     (≈ 1 day at 15-minute resolution).
   - Build a rolling z-score and mark points where the z-score exceeds a
     threshold (e.g. 4) as candidate leaks or sensor spikes.
   - Replace those spikes with a local rolling median to obtain a cleaned signal
     `consumption_clean`.
   - Keep diagnostic columns such as `zscore_pos` and `is_leak` so we can
     visualise where the algorithm decided to clamp the series.

   This step removes obviously unrealistic surges without touching the rest of
   the pattern.

 **Small-gap interpolation on a regular grid**

   - Align each station to a regular 15-minute grid.
   - Avoid inventing data during structural closures by using `struct_zero` as
     a mask: interpolation is only allowed when the station is expected to be
     “open”.
   - Interpolate **only very short gaps** (up to `max_gap_steps` consecutive
     missing steps) using time-based interpolation.
   - Longer gaps remain as missing and will be handled later by the
     sliding-window builder in the training notebook.

The output of `preprocess` is a cleaned panel:

   - `timestamp` (UTC, 15-minute grid)
   - `station`
   - `consumption` (original, clipped at zero, with small gaps interpolated)
   - `consumption_clean` (same scale, but with unrealistic surges smoothed)
   - `zscore_pos` and `is_leak` (optional diagnostics)



In [ ]:
import numpy as np
import pandas as pd

def preprocess_for_model(
    df_long_raw: pd.DataFrame,
    freq: str = "15min",
    max_gap_steps: int = 2,
    z_threshold: float = 4.0,
    roll_window: int = 96,  # ~1 day at 15min
    med_window: int = 24,   # shorter window for local median
) -> pd.DataFrame:
    """
    Light preprocessing for the demo:
      - Ensure timestamp / station / consumption exist
      - Normalize timestamp to UTC and sort
      - Per station:
          * enforce regular 15-min grid
          * drop duplicate timestamps
          * clip negative values to 0
          * interpolate small gaps
          * detect large positive surges via rolling z-score
          * clamp flagged surges to a rolling median
      - Add:
          * consumption_clean
          * zscore_pos
          * is_leak

    Returns:
      df_pre: long dataframe with columns at least:
              ['timestamp', 'station', 'consumption',
               'consumption_clean', 'zscore_pos', 'is_leak']
    """
    df = df_long_raw.copy()

    required_cols = {"timestamp", "station", "consumption"}
    missing = required_cols - set(df.columns)
    if missing:
        raise KeyError(f"Missing required columns in df_long: {missing}")

    df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True, errors="coerce")
    df = df.dropna(subset=["timestamp"])
    df = df.sort_values(["station", "timestamp"], kind="mergesort").reset_index(drop=True)

    def _clean_station(g: pd.DataFrame) -> pd.DataFrame:
        st = g["station"].iloc[0]

        # sort + index by timestamp
        g = g.sort_values("timestamp").set_index("timestamp")

        # remove duplicate timestamps (keep last)
        g = g[~g.index.duplicated(keep="last")]

        # enforce fixed time grid
        g = g.asfreq(freq)

        # original consumption as float
        vals = pd.to_numeric(g["consumption"], errors="coerce").astype("float32")

        # clip negatives: no negative flow in this context
        vals = vals.clip(lower=0)

        # interpolate only short internal gaps
        vals_interp = vals.interpolate(
            method="time",
            limit=max_gap_steps,
            limit_area="inside",
            limit_direction="forward",
        )

        # rolling mean / std for z-score (based on interpolated series)
        rm = vals_interp.rolling(roll_window, min_periods=1).mean()
        rs = vals_interp.rolling(roll_window, min_periods=1).std()

        # avoid division by zero
        z = (vals_interp - rm) / (rs + 1e-6)
        z_pos = z.clip(lower=0)

        # flag strong positive surges
        is_leak = (z_pos > z_threshold)

        # local median baseline (shorter window)
        gm = vals_interp.rolling(med_window, min_periods=1).median()

        # clamp surges to local median
        clean = np.where(is_leak, gm, vals_interp)

        g["consumption"] = vals_interp.astype("float32")
        g["consumption_clean"] = clean.astype("float32")
        g["zscore_pos"] = z_pos.astype("float32")
        g["is_leak"] = is_leak.astype("int8")

        out = g.reset_index()
        out["station"] = st
        return out

    df_pre = (
        df
        .groupby("station", group_keys=False)
        .apply(_clean_station)
        .reset_index(drop=True)
    )

    return df_pre


In [ ]:
import json


df_pre = preprocess_for_model(df_long)
display(df_pre.head())

output_path = PROCESSED_DIR / "df_pre.parquet"
manifest_path = PROCESSED_DIR / "df_pre_manifest.json"

df_pre.to_parquet(output_path, index=False)

manifest = {
    "rows": int(len(df_pre)),
    "stations": int(df_pre["station"].nunique()),
    "timestamp_min_utc": str(df_pre["timestamp"].min()),
    "timestamp_max_utc": str(df_pre["timestamp"].max()),
    "columns": list(df_pre.columns),
}
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

print(f"Saved preprocessed dataset to: {output_path}")
print(f"Saved manifest to: {manifest_path}")
manifest
